In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

data_dir = "../../data/"

target = pd.read_csv(data_dir + "train.csv")

book = pd.read_parquet(data_dir + "book_train.parquet")
book["stock_id"] = book["stock_id"].astype("int16")

trade = pd.read_parquet(data_dir + "trade_train.parquet")
trade["stock_id"] = trade["stock_id"].astype("int16")

In [2]:

# 两档总挂单量。注意：算 WAP1 不用这个，用的是一档的买卖量之和
book["tot_size"] = book["bid_size1"] + book["bid_size2"] + book["ask_size1"] + book["ask_size2"]

# WAP（加权中间价）：买价乘卖量，卖价乘买量，权重是交叉的
# 卖单多，说明价格更可能往下走，所以 WAP 更靠近买价；买单多则更靠近卖价
book["wap1"] = ((book["bid_price1"] * book["ask_size1"] + book["ask_price1"] * book["bid_size1"])
             / (book["bid_size1"] + book["ask_size1"])).astype("float32")

book["wap2"] = ((book["bid_price2"] * book["ask_size2"] + book["ask_price2"] * book["bid_size2"])
             / (book["bid_size2"] + book["ask_size2"])).astype("float32")

book["wap12"] = ((book["bid_price1"] * book["ask_size1"] + book["ask_price1"] * book["bid_size1"] + 
            book["bid_price2"] * book["ask_size2"] + book["ask_price2"] * book["bid_size2"])
             / book["tot_size"]).astype("float32")

for k in ["1", "2", "12"]:
    book[f"log_wap{k}"] = np.log(book[f"wap{k}"])

log_cols = ["log_wap1", "log_wap2", "log_wap12"]
ret_cols = ["log_ret1", "log_ret2", "log_ret12"]

book[ret_cols] = book.groupby(["stock_id", "time_id"], observed=True)[log_cols].diff().to_numpy()


In [3]:
def realized_vol(series):
    return np.sqrt(np.sum(np.square(series)))

rv = book.groupby(by = ["stock_id", "time_id"], observed=True)[["log_ret1", "log_ret2", "log_ret12"]].agg(realized_vol).reset_index()

rv.rename(columns={"log_ret1":"rv1", "log_ret2":"rv2", "log_ret12":"rv12"}, inplace=True)

rv

,stock_id,time_id,rv1,rv2,rv12
0,0,5,0.004499,0.006999,0.004106
1,0,11,0.001204,0.002476,0.001507
2,0,16,0.002369,0.004801,0.002469
3,0,31,0.002574,0.003637,0.002709
4,0,62,0.001895,0.003257,0.001932
...,...,...,...,...,...
428927,126,32751,0.003691,0.005877,0.003620
428928,126,32753,0.004104,0.004991,0.003570
428929,126,32758,0.003118,0.006019,0.003488
428930,126,32763,0.003661,0.005362,0.003209


In [4]:
combined = pd.merge(target, rv, how = "outer", on = ["stock_id", "time_id"])
combined.isna().any()


stock_id    False
time_id     False
target      False
rv1         False
rv2         False
rv12        False
dtype: bool

In [5]:
def rmspe(y_true, y_pred):
    return np.sqrt(np.mean(((y_true - y_pred) / y_true) ** 2))

# baseline：直接用当前窗口的 RV 预测 target
rmspe(combined["target"], combined["rv1"])


0.34135441481106876

In [6]:
cols = ["rv1", "rv2", "rv12", "target"]

print("Pearson（原始值）")
print(combined[cols].corr()["target"])

print("Pearson（取 log 后）")
print(np.log(combined[cols]).corr()["target"])

print("Spearman（秩相关）")
print(combined[cols].corr(method="spearman")["target"])


Pearson（原始值）
rv1       0.873777
rv2       0.863870
rv12      0.865103
target    1.000000
Name: target, dtype: float64
Pearson（取 log 后）
rv1       0.910804
rv2       0.900791
rv12      0.900584
target    1.000000
Name: target, dtype: float64
Spearman（秩相关）
rv1       0.908099
rv2       0.899633
rv12      0.897520
target    1.000000
Name: target, dtype: float64


In [ ]:
def window_stats(df, col, keys=["stock_id", "time_id"]):
    g = df.groupby(keys, observed=True)[col]
    out = pd.DataFrame({
        "mean": g.mean(),
        "std":  g.std(ddof=0),
        "iqr":  g.quantile(0.75) - g.quantile(0.25),
        "last": g.last(),
        "max": g.max(),
    })
    out["zscore_last"] = ((out["last"] - out["mean"]) / out["std"]).where(out["std"] > 0, 0.0)
    out["std_mean"] = out["iqr"] / out["mean"]
    return out.drop(columns="last").add_prefix(f"{col}_")

In [13]:
book["spread"] = book["ask_price1"] - book["bid_price1"]
# book["imb1"] = book["ask_size1"] / 

grouped = book.groupby(["stock_id", "time_id"], observed=True)

fwap1_to1 = (grouped["wap1"].first() - 1.).abs().rename("fwap1_to1")
lwap1_to1 = (grouped["wap1"].last() - 1.).abs().rename("lwap1_to1")

In [21]:
# 1. 先逐行算出每个时刻的不平衡
book["imb1"] = ((book.bid_size1 - book.ask_size1)
                / (book.bid_size1 + book.ask_size1)).astype("float32")
book["imb12"] = ((book.bid_size1 + book.bid_size2 - book.ask_size1 - book.ask_size2)
                 / (book.bid_size1 + book.bid_size2 + book.ask_size1 + book.ask_size2)).astype("float32")
book["imb1_abs"] = book["imb1"].abs()
book["imb12_abs"] = book["imb12"].abs()

# 2. 再按窗口聚合，命名聚合会直接给好列名
imb_features = book.groupby(["stock_id", "time_id"], observed=True).agg(
    imb1_mean     =("imb1", "mean"),
    imb1_std      =("imb1", "std"),
    imb1_abs_mean =("imb1_abs", "mean"),
    imb12_mean    =("imb12", "mean"),
    imb12_std     =("imb12", "std"),
    imb12_abs_mean=("imb12_abs", "mean"),
)


In [ ]:
feature_stats = window_stats(book, "spread")

for feature_class in ["wap1", "wap2", "wap12", "tot_size", "log_ret1", "log_ret12"]:
    feature_stats = feature_stats.merge(window_stats(book, feature_class), 
    on=["stock_id", "time_id"], how="left", validate="one_to_one")

feature_stats


spread_mean  spread_std  spread_iqr  spread_max  \
stock_id time_id                                                    
0        5           0.000855    0.000212    0.000207    0.001396   
         11          0.000394    0.000157    0.000201    0.000903   
         16          0.000725    0.000163    0.000251    0.001149   
         31          0.000859    0.000278    0.000463    0.001619   
         62          0.000397    0.000130    0.000140    0.000792   
...                       ...         ...         ...         ...   
126      32751       0.000877    0.000234    0.000392    0.001567   
         32753       0.000707    0.000228    0.000345    0.001379   
         32758       0.000740    0.000239    0.000309    0.001284   
         32763       0.000531    0.000172    0.000198    0.001119   
         32767       0.000432    0.000125    0.000154    0.000823   

                  spread_zscore_last  spread_std_mean  wap1_mean  wap1_std  \
stock_id time_id                                                             
0        5                 -1.106809         0.241896   1.003725  0.000692   
         11                 0.686592         0.509198   1.000239  0.000261   
         16                -1.801277         0.346638   0.999542  0.000862   
         31                 1.068402         0.538351   0.998832  0.000754   
         62                -0.185681         0.352369   0.999619  0.000257   
...                              ...              ...        ...       ...   
126      32751              0.436031         0.446498   0.999582  0.000485   
         32753              2.489258         0.487279   1.002476  0.001262   
         32758              1.859070         0.417268   1.001082  0.000465   
         32763              1.124899         0.371990   1.001809  0.000456   
         32767              0.248336         0.357631   1.000271  0.000384   

                  wap1_iqr  wap1_max  ...  wap2_iqr  wap2_max  \
stock_id time_id                      ...                       
0        5        0.000543  1.004920  ...  0.000629  1.005124   
         11       0.000215  1.000834  ...  0.000302  1.001067   
         16       0.001143  1.000878  ...  0.000969  1.000876   
         31       0.000772  1.000412  ...  0.000763  1.000116   
         62       0.000406  1.000159  ...  0.000585  1.000249   
...                    ...       ...  ...       ...       ...   
126      32751    0.000506  1.000736  ...  0.000872  1.000788   
         32753    0.001746  1.006166  ...  0.001879  1.006178   
         32758    0.000643  1.002656  ...  0.000922  1.002849   
         32763    0.000621  1.002963  ...  0.000666  1.003009   
         32767    0.000451  1.001058  ...  0.000485  1.001306   

                  wap2_zscore_last  wap2_std_mean  wap12_mean  wap12_std  \
stock_id time_id                                                           
0        5                0.077194       0.000627    1.003727   0.000713   
         11               1.470572       0.000302    1.000236   0.000226   
         16              -2.104199       0.000969    0.999602   0.000846   
         31              -1.446316       0.000764    0.998743   0.000694   
         62               0.253420       0.000585    0.999657   0.000249   
...                            ...            ...         ...        ...   
126      32751           -1.248790       0.000872    0.999556   0.000494   
         32753            2.450564       0.001874    1.002565   0.001274   
         32758            3.097826       0.000921    1.001103   0.000510   
         32763           -0.886104       0.000665    1.001806   0.000467   
         32767           -2.555403       0.000485    1.000296   0.000401   

                  wap12_iqr  wap12_max  wap12_zscore_last  wap12_std_mean  
stock_id time_id                                                           
0        5         0.000453   1.004968           0.002508        0.000451  
         11        0.000260   1.000972    

In [15]:
feature_stats = feature_stats.merge(fwap1_to1, on=["stock_id", "time_id"], how="left", validate="one_to_one")
feature_stats = feature_stats.merge(lwap1_to1, on=["stock_id", "time_id"], how="left", validate="one_to_one")

feature_stats

spread_mean  spread_std  spread_iqr  spread_max  \
stock_id time_id                                                    
0        5           0.000855    0.000212    0.000207    0.001396   
         11          0.000394    0.000157    0.000201    0.000903   
         16          0.000725    0.000163    0.000251    0.001149   
         31          0.000859    0.000278    0.000463    0.001619   
         62          0.000397    0.000130    0.000140    0.000792   
...                       ...         ...         ...         ...   
126      32751       0.000877    0.000234    0.000392    0.001567   
         32753       0.000707    0.000228    0.000345    0.001379   
         32758       0.000740    0.000239    0.000309    0.001284   
         32763       0.000531    0.000172    0.000198    0.001119   
         32767       0.000432    0.000125    0.000154    0.000823   

                  spread_zscore_last  spread_std_mean  wap1_mean  wap1_std  \
stock_id time_id                                                             
0        5                 -1.106809         0.241896   1.003725  0.000692   
         11                 0.686592         0.509198   1.000239  0.000261   
         16                -1.801277         0.346638   0.999542  0.000862   
         31                 1.068402         0.538351   0.998832  0.000754   
         62                -0.185681         0.352369   0.999619  0.000257   
...                              ...              ...        ...       ...   
126      32751              0.436031         0.446498   0.999582  0.000485   
         32753              2.489258         0.487279   1.002476  0.001262   
         32758              1.859070         0.417268   1.001082  0.000465   
         32763              1.124899         0.371990   1.001809  0.000456   
         32767              0.248336         0.357631   1.000271  0.000384   

                  wap1_iqr  wap1_max  ...  wap2_zscore_last  wap2_std_mean  \
stock_id time_id                      ...                                    
0        5        0.000543  1.004920  ...          0.077194       0.000627   
         11       0.000215  1.000834  ...          1.470572       0.000302   
         16       0.001143  1.000878  ...         -2.104199       0.000969   
         31       0.000772  1.000412  ...         -1.446316       0.000764   
         62       0.000406  1.000159  ...          0.253420       0.000585   
...                    ...       ...  ...               ...            ...   
126      32751    0.000506  1.000736  ...         -1.248790       0.000872   
         32753    0.001746  1.006166  ...          2.450564       0.001874   
         32758    0.000643  1.002656  ...          3.097826       0.000921   
         32763    0.000621  1.002963  ...         -0.886104       0.000665   
         32767    0.000451  1.001058  ...         -2.555403       0.000485   

                  wap12_mean  wap12_std  wap12_iqr  wap12_max  \
stock_id time_id                                                
0        5          1.003727   0.000713   0.000453   1.004968   
         11         1.000236   0.000226   0.000260   1.000972   
         16         0.999602   0.000846   0.000954   1.000830   
         31         0.998743   0.000694   0.000647   1.000400   
         62         0.999657   0.000249   0.000396   1.000145   
...                      ...        ...        ...        ...   
126      32751      0.999556   0.000494   0.000613   1.000767   
         32753      1.002565   0.001274   0.001895   1.005977   
         32758      1.001103   0.000510   0.000736   1.002844   
         32763      1.001806   0.000467   0.000609   1.002966   
         32767      1.000296   0.000401   0.000427   1.001056   

                  wap12_zscore_last  wap12_std_mean  fwap1_to1  lwap1_to1  
stock_id time_id                                                           
0        5                 0.002508        0.000451   0.001434   0.003731  
         11     

In [ ]:
feature_stats = feature_stats.merge(imb_features, on=["stock_id", "time_id"], how="left", validate="one_to_one")
feature_stats

spread_mean  spread_std  spread_iqr  spread_max  \
stock_id time_id                                                    
0        5           0.000855    0.000212    0.000207    0.001396   
         11          0.000394    0.000157    0.000201    0.000903   
         16          0.000725    0.000163    0.000251    0.001149   
         31          0.000859    0.000278    0.000463    0.001619   
         62          0.000397    0.000130    0.000140    0.000792   
...                       ...         ...         ...         ...   
126      32751       0.000877    0.000234    0.000392    0.001567   
         32753       0.000707    0.000228    0.000345    0.001379   
         32758       0.000740    0.000239    0.000309    0.001284   
         32763       0.000531    0.000172    0.000198    0.001119   
         32767       0.000432    0.000125    0.000154    0.000823   

                  spread_zscore_last  spread_std_mean  wap1_mean  wap1_std  \
stock_id time_id                                                             
0        5                 -1.106809         0.241896   1.003725  0.000692   
         11                 0.686592         0.509198   1.000239  0.000261   
         16                -1.801277         0.346638   0.999542  0.000862   
         31                 1.068402         0.538351   0.998832  0.000754   
         62                -0.185681         0.352369   0.999619  0.000257   
...                              ...              ...        ...       ...   
126      32751              0.436031         0.446498   0.999582  0.000485   
         32753              2.489258         0.487279   1.002476  0.001262   
         32758              1.859070         0.417268   1.001082  0.000465   
         32763              1.124899         0.371990   1.001809  0.000456   
         32767              0.248336         0.357631   1.000271  0.000384   

                  wap1_iqr  wap1_max  ...  wap12_std_mean  fwap1_to1  \
stock_id time_id                      ...                              
0        5        0.000543  1.004920  ...        0.000451   0.001434   
         11       0.000215  1.000834  ...        0.000260   0.000055   
         16       0.001143  1.000878  ...        0.000955   0.000326   
         31       0.000772  1.000412  ...        0.000647   0.000316   
         62       0.000406  1.000159  ...        0.000396   0.000079   
...                    ...       ...  ...             ...        ...   
126      32751    0.000506  1.000736  ...        0.000614   0.000134   
         32753    0.001746  1.006166  ...        0.001890   0.001259   
         32758    0.000643  1.002656  ...        0.000736   0.001128   
         32763    0.000621  1.002963  ...        0.000608   0.000562   
         32767    0.000451  1.001058  ...        0.000426   0.000188   

                  lwap1_to1  imb1_mean  imb1_std  imb1_abs_mean  imb12_mean  \
stock_id time_id                                                              
0        5         0.003731  -0.006357  0.708949       0.599749    0.013154   
         11        0.000306   0.167165  0.702103       0.634449    0.177003   
         16        0.002397  -0.105494  0.583874       0.474173    0.030383   
         31        0.002509   0.056378  0.709451       0.644306   -0.119307   
         62        0.000077   0.088670  0.674933       0.570008    0.271247   
...                     ...        ...       ...            ...         ...   
126      32751     0.000393   0.331854  0.626346       0.617483    0.198069   
         32753     0.005710   0.230805  0.686361       0.656237    0.393306   
         32758     0.002656   0.694757  0.419117       0.750202    0.644336   
         32763     0.000818  -0.287771  0.551020       0.513716   -0.227392   
         32767     0.000539  -0.117071  0.531778       0.422530   -0.058036   

                  imb12_std  imb12_abs_mean  n_quotes  
stock_id time_id                                       
0        5         0.606090 

In [31]:
n_quotes = grouped.size().rename("n_quotes")
n_trades = trade.groupby(["stock_id", "time_id"]).size().rename("n_trades")

trade_ratio = pd.merge(n_quotes, n_trades, on = ["stock_id", "time_id"], how = "left")
trade_ratio["trade_ratio"] = trade_ratio["n_quotes"] / trade_ratio["n_trades"]
trade_ratio.fillna(0)
trade_ratio

n_quotes  n_trades  trade_ratio
stock_id time_id                                 
0        5             302      40.0     7.550000
         11            200      30.0     6.666667
         16            188      25.0     7.520000
         31            120      15.0     8.000000
         62            176      22.0     8.000000
...                    ...       ...          ...
126      32751         310      37.0     8.378378
         32753         223      43.0     5.186047
         32758         256      35.0     7.314286
         32763         399      80.0     4.987500
         32767         217      36.0     6.027778

[428932 rows x 3 columns]

In [33]:
trade["log_price"] = np.log(trade["price"])
trade["log_ret"] = trade["log_price"].diff()
trade_rv = trade.groupby(by = ["stock_id", "time_id"], observed=True)[["log_ret"]].agg(realized_vol)
trade_rv.rename(columns={"log_ret":"trade_rv"}, inplace=True)

trade_rv

trade_rv
stock_id time_id          
0        5        0.002006
         11       0.003920
         16       0.002004
         31       0.002995
         62       0.002230
...                    ...
126      32751    0.002385
         32753    0.002994
         32758    0.005382
         32763    0.002174
         32767    0.001118

[428913 rows x 1 columns]

In [35]:
trade_features = window_stats(trade, "price")
""
for feature_class in ["size", "price", "log_ret"]:
    trade_features = trade_features.merge(window_stats(trade, feature_class), 
    on=["stock_id", "time_id"], how="left", validate="one_to_one")

trade_features


price_mean_x  price_std_x  price_iqr_x  price_max_x  \
stock_id time_id                                                        
0        5            1.003722     0.000571     0.000651     1.004680   
         11           1.000206     0.000298     0.000397     1.000828   
         16           0.999204     0.000914     0.001628     1.000455   
         31           0.999020     0.000704     0.001038     1.000231   
         62           0.999618     0.000178     0.000228     1.000023   
...                        ...          ...          ...          ...   
126      32751        0.999289     0.000455     0.000670     1.000522   
         32753        1.002344     0.001217     0.001965     1.005586   
         32758        1.000673     0.000403     0.000476     1.001837   
         32763        1.001855     0.000385     0.000461     1.002604   
         32767        1.000230     0.000327     0.000420     1.000857   

                  price_zscore_last_x  price_std_mean_x   size_mean  \
stock_id time_id                                                      
0        5                  -0.133823          0.000648   79.475000   
         11                  1.411326          0.000397   42.966667   
         16                 -1.669901          0.001629   86.440000   
         31                 -1.488465          0.001039  130.800000   
         62                  0.737162          0.000228   81.409091   
...                               ...               ...         ...   
126      32751              -1.382004          0.000671   69.459459   
         32753               2.664665          0.001961   54.023256   
         32758               2.888665          0.000476  106.857143   
         32763              -2.512029          0.000460  117.362500   
         32767              -2.351383          0.000420  147.916667   

                    size_std  size_iqr  size_max  ...  price_iqr_y  \
stock_id time_id                                  ...                
0        5        116.886053    107.75       499  ...     0.000651   
         11        76.507291     26.75       280  ...     0.000397   
         16       111.292077    122.00       391  ...     0.001628   
         31       139.917690    131.50       450  ...     0.001038   
         62       115.203638    112.50       341  ...     0.000228   
...                      ...       ...       ...  ...          ...   
126      32751    101.336633     97.00       528  ...     0.000670   
         32753     70.198883     98.50       238  ...     0.001965   
         32758    113.524860    133.50       498  ...     0.000476   
         32763    140.108373    193.50       738  ...     0.000461   
         32767    127.832872    123.50       500  ...     0.000420   

                  price_max_y  price_zscore_last_y  price_std_mean_y  \
stock_id time_id                                                       
0        5           1.004680            -0.133823          0.000648   
         11          1.000828             1.411326          0.000397   
         16          1.000455            -1.669901          0.001629   
         31          1.000231            -1.488465          0.001039   
         62          1.000023             0.737162          0.000228   
...                       ...                  ...               ...   
126      32751       1.000522            -1.382004          0.000671   
         32753       1.005586             2.664665          0.001961   
         32758       1.001837             2.888665          0.000476   
         32763       1.002604            -2.512029          0.000460   
         32767       1.000857            -2.351383          0.000420   

                  log_ret_mean  log_ret_std  log_ret_iqr  log_ret_max  \
stock_id time_id                                                        
0        5            0.000034     0.000319     0.000517     0.000610   
         11          -0.000100     0.000709     0.000200     0.000402   
    

In [36]:
tot_volume = trade.groupby(["stock_id", "time_id"])["size"].sum()
tot_volume

stock_id  time_id
0         5          3179
          11         1289
          16         2161
          31         1962
          62         1791
                     ... 
126       32751      2570
          32753      2323
          32758      3740
          32763      9389
          32767      5325
Name: size, Length: 428913, dtype: int32

In [2]:
keys = ["stock_id", "time_id", "seconds_in_bucket"]

# 1. 上一个时刻的 mid 和半价差（成交前的报价）
quotes = book[keys].copy()
quotes["mid"] = ((book["bid_price1"] + book["ask_price1"]) / 2).astype("float32")
quotes["half_spread"] = ((book["ask_price1"] - book["bid_price1"]) / 2).astype("float32")

grp = quotes.groupby(["stock_id", "time_id"], observed=True)
quotes["mid_prev"] = grp["mid"].shift(1)
quotes["hs_prev"] = grp["half_spread"].shift(1)

# 2. 以 trade 为左表合并
tq = trade.merge(quotes, on=keys, how="left", validate="one_to_one")

# 窗口第一行没有上一个时刻，就用同一秒的盘口
mid_ref = tq["mid_prev"].fillna(tq["mid"])
hs_ref = tq["hs_prev"].fillna(tq["half_spread"])

# 3. 成交价的相对位置：-1 = bid，0 = mid，+1 = ask
tq["x"] = (tq["price"] - mid_ref) / hs_ref

THRESH = 0.5
tq["is_buy"] = tq["x"] >= THRESH
tq["is_sell"] = tq["x"] <= -THRESH
tq["buy_vol"] = tq["size"] * tq["is_buy"]
tq["sell_vol"] = tq["size"] * tq["is_sell"]
tq["abs_x"] = tq["x"].abs()
tq["sweep"] = tq["abs_x"] > 1.5          # 成交价远超出一档，说明吃穿了好几档

# 4. 按窗口聚合
side_feats = tq.groupby(["stock_id", "time_id"], observed=True).agg(
    n_buy=("is_buy", "sum"),
    n_sell=("is_sell", "sum"),
    buy_vol=("buy_vol", "sum"),
    sell_vol=("sell_vol", "sum"),
    abs_x_mean=("abs_x", "mean"),
    sweep_frac=("sweep", "mean"),
)

# 5. 比例特征（分母为 0 时是 NaN，LightGBM 可以处理）
side_feats["buy_frac"] = side_feats["n_buy"] / (side_feats["n_buy"] + side_feats["n_sell"])
side_feats["buy_vol_frac"] = side_feats["buy_vol"] / (side_feats["buy_vol"] + side_feats["sell_vol"])

side_feats = side_feats[["buy_frac", "buy_vol_frac", "abs_x_mean", "sweep_frac"]]

del quotes, tq
side_feats


buy_frac  buy_vol_frac  abs_x_mean  sweep_frac
stock_id time_id                                                
0        5        0.647059      0.567679    0.902439    0.000000
         11       0.590909      0.458834    0.760597    0.000000
         16       0.200000      0.223513    0.902782    0.040000
         31       0.363636      0.416537    0.912570    0.066667
         62       0.588235      0.392576    0.888332    0.045455
...                    ...           ...         ...         ...
126      32751    0.413793      0.435189    0.813497    0.000000
         32753    0.722222      0.467582    0.909978    0.046512
         32758    0.354839      0.109877    0.965501    0.028571
         32763    0.383562      0.617441    1.044486    0.075000
         32767    0.387097      0.331271    0.961673    0.027778

[428913 rows x 4 columns]